# 03 – Toxicity Model

Train endpoint-specific Random Forest models using Tox21 molecular features. The workflow uses missing-label-aware training, class-balanced learning, scaffold-aware validation when feasible, PR-AUC/ROC-AUC evaluation, calibration, and SHAP interpretation.

**Decision-support boundary:** predictions prioritize follow-up work; they do not establish clinical toxicity or biological causality.

In [1]:
from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent
if not (ROOT/"data").exists(): ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.tox21_model import make_model, evaluate_binary
from src.preprocessing import TOX21_ENDPOINTS

DATA = ROOT/"data/processed/tox21"
X = pd.read_parquet(DATA/"X_tox21.parquet")
Y = pd.read_parquet(DATA/"Y_tox21.parquet")
meta = pd.read_parquet(DATA/"tox21_metadata.parquet")

## Split strategy

A scaffold split is preferred because random splitting can place near-neighbor chemistry in both train and test. This cell attempts a Murcko-scaffold split and falls back to a stratified random split if necessary.

In [2]:
def scaffold_split(smiles, test_size=0.2, valid_size=0.2, seed=42):
    try:
        from rdkit import Chem
        from rdkit.Chem.Scaffolds import MurckoScaffold
        groups = {}
        for idx, smi in smiles.items():
            mol = Chem.MolFromSmiles(smi) if pd.notna(smi) else None
            scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol) if mol else f"invalid_{idx}"
            groups.setdefault(scaf, []).append(idx)
        rng = np.random.default_rng(seed)
        keys = list(groups); rng.shuffle(keys)
        n = len(smiles); target_test=int(n*test_size); target_valid=int(n*valid_size)
        test, valid, train = [], [], []
        for k in keys:
            if len(test) < target_test: test += groups[k]
            elif len(valid) < target_valid: valid += groups[k]
            else: train += groups[k]
        return train, valid, test, "scaffold"
    except Exception:
        idx = np.arange(len(smiles))
        train, temp = train_test_split(idx, test_size=test_size+valid_size, random_state=seed)
        valid, test = train_test_split(temp, test_size=0.5, random_state=seed)
        return list(train), list(valid), list(test), "stratified_random"

train_idx, valid_idx, test_idx, split_method = scaffold_split(meta["smiles"])
print(split_method, len(train_idx), len(valid_idx), len(test_idx))

[01:25:41] WARNING: not removing hydrogen atom without neighbors
[01:25:42] Explicit valence for atom # 8 Al, 6, is greater than permitted
[01:25:42] Explicit valence for atom # 3 Al, 6, is greater than permitted
[01:25:42] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:25:43] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:25:43] Explicit valence for atom # 9 Al, 6, is greater than permitted
[01:25:43] Explicit valence for atom # 5 Al, 6, is greater than permitted
[01:25:44] Explicit valence for atom # 16 Al, 6, is greater than permitted
[01:25:44] Explicit valence for atom # 20 Al, 6, is greater than permitted


scaffold 3152 1568 3111


In [3]:
models, results = {}, []

for endpoint in TOX21_ENDPOINTS:
    y = Y[endpoint]
    tr = [i for i in train_idx if pd.notna(y.iloc[i])]
    te = [i for i in test_idx if pd.notna(y.iloc[i])]
    if len(tr) < 20 or y.iloc[tr].nunique() < 2 or not te:
        continue

    model = make_model(random_state=42)
    model.fit(X.iloc[tr], y.iloc[tr].astype(int))
    p = model.predict_proba(X.iloc[te])[:,1]
    row = evaluate_binary(y.iloc[te], p)
    row["endpoint"] = endpoint
    models[endpoint] = model
    results.append(row)

results = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
display(results)

,n,accuracy,f1,roc_auc,pr_auc,endpoint
1,2763,0.985161,0.700730,0.887342,0.679690,NR-AR-LBD
10,2412,0.922056,0.507853,0.896043,0.608628,SR-MMP
0,2921,0.982198,0.648649,0.807563,0.575522,NR-AR
2,2708,0.947194,0.430279,0.879299,0.489366,NR-AhR
4,2572,0.912908,0.300000,0.701001,0.414099,NR-ER
5,2830,0.962191,0.318471,0.808127,0.409673,NR-ER-LBD
7,2438,0.863823,0.242009,0.739136,0.384954,SR-ARE
6,2671,0.977162,0.031746,0.777866,0.328146,NR-PPAR-gamma
3,2454,0.975550,0.230769,0.821381,0.290972,NR-Aromatase
11,2751,0.954198,0.073529,0.843142,0.290848,SR-p53


## Calibration

For at least one endpoint, isotonic calibration can make probabilities more useful for risk-ranking. Calibration quality must be evaluated on data not used to fit the calibration model.

In [4]:
if not results.empty:
    best = results.iloc[0]["endpoint"]
    y = Y[best]
    tr = [i for i in train_idx if pd.notna(y.iloc[i])]
    base = make_model(random_state=42)
    calibrated = CalibratedClassifierCV(base, method="isotonic", cv=3)
    calibrated.fit(X.iloc[tr], y.iloc[tr].astype(int))
    models[best + "__calibrated"] = calibrated
    print("Calibrated endpoint:", best)

Calibrated endpoint: NR-AR-LBD


In [5]:
MODEL_DIR = ROOT/"models/tox21"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for endpoint, model in models.items():
    name = endpoint.replace("/", "_").replace("-", "_")
    joblib.dump(model, MODEL_DIR/f"{name}.joblib")

results.to_csv(MODEL_DIR/"tox21_model_performance.csv", index=False)
with open(MODEL_DIR/"metadata.json","w") as f:
    json.dump({"split_method":split_method,"random_state":42,
               "endpoints":list(models),"intended_use":"Early R&D safety triage",
               "limitations":["Tox21 is pathway-level","missing labels","not causal"]}, f, indent=2)
print("Models saved.")

Models saved.


## SHAP interpretation

Use SHAP with the fitted Random Forest to explain global feature contributions and selected molecules. Fingerprint features can be highly numerous; prioritize aggregate importance and representative examples.

**TODO:** map important fingerprint bits back to molecular substructures and review them with a chemist/toxicologist.

In [6]:
try:
    import shap
    endpoint = results.iloc[0]["endpoint"]
    pipe = models[endpoint]
    sample = X.sample(min(200, len(X)), random_state=42)
    imp = pipe.named_steps["imputer"].transform(sample)
    rf = pipe.named_steps["model"]
    explainer = shap.TreeExplainer(rf)
    sv = explainer.shap_values(imp)
    sv = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv, imp, feature_names=X.columns, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(ROOT/f"reports/figures/shap_{endpoint.replace('-','_')}.png", dpi=200, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("SHAP section skipped:", e)

SHAP section skipped: No module named 'shap'
